In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/telco_customer_churn.csv")
df = pd.read_csv(DATA_PATH)

raw_rows = len(df)
raw_duplicates = df["customerID"].duplicated().sum()
raw_missing_total = df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True).isna().sum()

df = df.drop_duplicates(subset="customerID").copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_total = df["TotalCharges"].isna().sum()
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0–12 months", "13–24 months", "25–48 months", "49–72 months"]
)

churn_rate = df["ChurnFlag"].mean()


### Data Quality Check

In [ ]:

quality = pd.DataFrame({
    "Measure": [
        "Rows before cleaning",
        "Duplicate customer records removed",
        "Missing TotalCharges before cleaning",
        "Unique customers after cleaning",
        "Missing TotalCharges after cleaning"
    ],
    "Value": [
        raw_rows,
        raw_duplicates,
        raw_missing_total,
        len(df),
        int(df["TotalCharges"].isna().sum())
    ]
])
quality


# Key Findings

## Finding 1 — The first year is the highest-risk period

Customer churn is dramatically higher among newer customers. The first 12 months are the most vulnerable period, making early onboarding and engagement the clearest retention opportunity.


In [ ]:

tenure_rates = df.groupby("TenureGroup", observed=True)["ChurnFlag"].mean().mul(100)

fig, ax = plt.subplots(figsize=(9, 5))
tenure_rates.plot(kind="bar", ax=ax)
ax.set_title("Churn is highest during the first year")
ax.set_xlabel("Customer tenure")
ax.set_ylabel("Customers who churn (%)")
ax.tick_params(axis="x", rotation=0)
ax.grid(False)
plt.tight_layout()
plt.show()


## Finding 2 — Month-to-month contracts carry the greatest retention risk

Customers without a longer commitment churn more often than customers on one- or two-year contracts. This makes contract migration a practical retention lever, provided the offer is valuable and transparent.


In [ ]:

contract_rates = df.groupby("Contract")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
contract_rates.plot(kind="bar", ax=ax)
ax.set_title("Month-to-month customers have the highest churn rate")
ax.set_xlabel("Contract type")
ax.set_ylabel("Customers who churn (%)")
ax.tick_params(axis="x", rotation=0)
ax.grid(False)
plt.tight_layout()
plt.show()


## Finding 3 — Fiber-optic customers show elevated churn

Fiber-optic customers have a higher churn rate than DSL customers in this dataset. Because fiber customers can also carry higher monthly charges, the business should investigate service quality, pricing, installation experience, and perceived value before assuming the problem is purely contractual.


In [ ]:

internet_rates = df.groupby("InternetService")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
internet_rates.plot(kind="bar", ax=ax)
ax.set_title("Fiber-optic customers show higher churn")
ax.set_xlabel("Internet service")
ax.set_ylabel("Customers who churn (%)")
ax.tick_params(axis="x", rotation=0)
ax.grid(False)
plt.tight_layout()
plt.show()


## Finding 4 — Electronic-check customers are a higher-risk payment segment

Customers using electronic checks churn more frequently than customers using automatic bank transfer or credit-card payments. This pattern may reflect payment friction, customer preferences, or differences in the types of customers who choose each method, so it is best treated as a retention signal rather than a proven cause.


In [ ]:

payment_rates = df.groupby("PaymentMethod")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
payment_rates.plot(kind="bar", ax=ax)
ax.set_title("Electronic-check customers have higher churn")
ax.set_xlabel("Payment method")
ax.set_ylabel("Customers who churn (%)")
ax.tick_params(axis="x", rotation=20)
ax.grid(False)
plt.tight_layout()
plt.show()


## Additional View — Monthly charges and churn

Higher monthly charges are associated with a greater concentration of churn in this dataset. This is useful for identifying customers who may need stronger value communication or targeted retention offers.


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="MonthlyCharges", by="Churn", ax=ax)
ax.set_title("Monthly charges are higher among customers who churn")
ax.set_xlabel("Churn status")
ax.set_ylabel("Monthly charges")
plt.suptitle("")
ax.grid(False)
plt.tight_layout()
plt.show()


## Additional View — Support availability

Customers without technical support show a higher churn rate than customers with support. This suggests that service assistance may be worth testing as part of a targeted retention program.


In [ ]:

support_rates = df.groupby("TechSupport")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
support_rates.plot(kind="bar", ax=ax)
ax.set_title("Customers without technical support churn more often")
ax.set_xlabel("Technical support")
ax.set_ylabel("Customers who churn (%)")
ax.tick_params(axis="x", rotation=20)
ax.grid(False)
plt.tight_layout()
plt.show()


# Recommendations

### 1. Build a first-year retention program
Prioritize onboarding, proactive check-ins, service education, and early satisfaction surveys during the first 12 months. The first-year churn rate is the strongest risk signal in this analysis.

### 2. Encourage longer-term contracts with value-based offers
Target month-to-month customers with transparent upgrade incentives, flexible annual plans, or benefits that make a longer commitment attractive without relying on heavy discounts.

### 3. Investigate the fiber and electronic-check segments
Run focused customer-experience research for fiber customers and review billing/payment friction among electronic-check users. Use the results to test targeted service improvements, payment options, or retention offers.


# Appendix — Technical Cleaning & EDA Code

The technical workflow below is included for reviewers who want to reproduce or extend the analysis. The business-facing sections above intentionally keep the presentation simple.


In [ ]:

# Full technical checks
print("Shape after cleaning:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nDuplicate customer IDs:", df["customerID"].duplicated().sum())

print("\nOverall churn rate:", round(df["ChurnFlag"].mean() * 100, 2), "%")

print("\nChurn by tenure group:")
print((df.groupby("TenureGroup", observed=True)["ChurnFlag"].mean() * 100).round(2))

print("\nChurn by contract:")
print((df.groupby("Contract")["ChurnFlag"].mean() * 100).round(2))

print("\nChurn by internet service:")
print((df.groupby("InternetService")["ChurnFlag"].mean() * 100).round(2))

print("\nChurn by payment method:")
print((df.groupby("PaymentMethod")["ChurnFlag"].mean() * 100).round(2))

print("\nChurn by technical support:")
print((df.groupby("TechSupport")["ChurnFlag"].mean() * 100).round(2))

# Export cleaned data for optional downstream work
clean_export = df.drop(columns=["ChurnFlag", "TenureGroup"])
clean_export.to_csv("../data/telco_customer_churn_clean.csv", index=False)
print("\nCleaned dataset exported.")
